# Transitivité stochastique

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import binom

## Concept

Si P(i bat j) ≥ 0.5 et P(j bat k) ≥ 0.5, alors on doit avoir P(i bat k) ≥ max(P(i bat j), P(j bat k))

Le problème c'est : comment savoir si la différence de score β entre le modèle rang 3 et le modèle rang 5 est réelle ou juste due au hasard ?

Le test de puissance répond exactement à ça : quel est le nombre minimum de comparaisons nécessaire pour détecter une différence réelle entre rang 3 et rang 5, avec 80% de chances de la détecter (puissance) et moins de 5% de faux positifs (α = 0.05) ?

Si ces deux modèles ne se sont jamais affrontés directement dans les données il est impossible de conclure sur cette paire par test binomial direct.
C'est précisément pourquoi le modèle Bradley-Terry est utile : il estime les β via les affrontements indirects (transitifs).

La différence β₃ - β₅ est l'effet que l'on cherches à détecter avec β₃ et β₅ les scores des deux modèles.

Plus cette différence est petite, plus il faut de comparaisons pour la détecter statistiquement.

Test de puissance pour une proportion :

n = (z_α + z_β)² / (p₁ - p₂)²

Où p₁ et p₂ sont les probabilités de victoire dérivées des β via Bradley-Terry.

Et

p₃₅ = 1 / (1 + exp(-(β₃ - β₅)))
C'est simplement la fonction sigmoïde appliquée à la différence des β.


Le calcul de puissance donne un résultat du nombre minimum de comparaisons nécessaire.

Si les modèles rang 3 et rang 5 ont bien été comparés au moins tant de fois : le classement est statistiquement fiable à ces rangs.
Si non : impossible de distinguer le rang 3 du rang 5 avec confiance.

### Etapes

* Filtrer sur les 20 modèles les plus comparés
* Estimer les β sur ce sous-ensemble depuis le classement global (exercice 2.1)
* Calculer le nombre minimum de comparaisons nécessaire (calcul de puissance sur p₃₅ = σ(β₃ - β₅) et p₅₃ = 1 - p₃₅)
* Vérifier que rang 3 et rang 5 ont bien ce minimum dans tes données

### Justifications

Le dataset Compar:IA contient des centaines de modèles, mais la majorité n'ont que quelques comparaisons. Le modèle Bradley-Terry (et ses extensions) nécessite suffisamment de données pour estimer des scores fiables. Avec trop peu de comparaisons, les paramètres β sont instables et les intervalles de confiance explosent.
En prenant les 20 modèles les plus comparés nous aurons :

* un graphe connexe (tout le monde a affronté tout le monde, au moins indirectement)
* des estimations β stables
* la compétition est la plus dense

Nous cherchons les β qui maximisent la probabilité d'observer toutes les comparaisons du dataset. C'est un problème d'optimisation convexe, résolu itérativement. Le résultat : chaque modèle reçoit un score β qui résume sa "force" globale déduite de tous ses matchs.

Tester la transitivité sur le sous-graphe des top 20, c'est donc aussi valider empiriquement que les données sont cohérentes avec les hypothèses du modèle Bradley-Terry : si nous trouvons des violations massives (A bat B, B bat C, mais C bat A dans les données brutes), le modèle Bradley-Terry est mal adapté.

In [2]:
df = pd.read_json("../data/comparia-votes/votes.jsonl", lines=True)

COL_A = "model_a_name"
COL_B = "model_b_name"
COL_WINNER = "chosen_model_name"

df[COL_A] = df[COL_A].str.strip()
df[COL_B] = df[COL_B].str.strip()
df[COL_WINNER] = df[COL_WINNER].str.strip()


top20_models = (
    pd.concat([df[COL_A], df[COL_B]])
    .value_counts()
    .head(20)
    .index.tolist()
)

df_top20 = df[
    df[COL_A].isin(top20_models) & df[COL_B].isin(top20_models)
].copy()

print(f"\nNombre de votes dans le top20 : {len(df_top20)}")


def fit_bradley_terry(df_pairs, col_winner=COL_WINNER, col_a=COL_A, col_b=COL_B):
    models = sorted(set(df_pairs[col_a]) | set(df_pairs[col_b]))
    idx = {m: i for i, m in enumerate(models)}
    n = len(models)

    wins = np.zeros((n, n))
    for _, row in df_pairs.iterrows():
        w = row[col_winner]
        a, b = row[col_a], row[col_b]
        if w == a:
            wins[idx[a], idx[b]] += 1
        elif w == b:
            wins[idx[b], idx[a]] += 1
        

    def neg_log_lik(beta):
        nll = 0.0
        for i in range(n):
            for j in range(n):
                if wins[i, j] > 0:
                    nll -= wins[i, j] * np.log(expit(beta[i] - beta[j]) + 1e-12)
        return nll

    beta0  = np.zeros(n)
    bounds = [(0, 0)] + [(None, None)] * (n - 1)
    res    = minimize(neg_log_lik, beta0, method="L-BFGS-B", bounds=bounds)

    result = (
        pd.DataFrame({"model": models, "beta": res.x})
        .sort_values("beta", ascending=False)
        .reset_index(drop=True)
    )
    result["rank"] = result.index + 1
    return result


ranking = fit_bradley_terry(df_top20)
print("\nClassement global (top 20) :")
print(ranking[["rank", "model", "beta"]].to_string(index=False))

beta3 = ranking.loc[ranking["rank"] == 3, "beta"].values[0]
beta5 = ranking.loc[ranking["rank"] == 5, "beta"].values[0]
model3 = ranking.loc[ranking["rank"] == 3, "model"].values[0]
model5 = ranking.loc[ranking["rank"] == 5, "model"].values[0]

print(f"\nRang 3 : {model3}  β = {beta3:.4f}")
print(f"Rang 5 : {model5}  β = {beta5:.4f}")


def min_comparisons_power(beta_i, beta_j, alpha=0.05, power=0.80):
    p_ij = expit(beta_i - beta_j)
    print(f"\np35 = σ(β3 − β5) = {p_ij:.4f}")
    print(f"p53 = 1 − p35    = {1 - p_ij:.4f}")

    for n in range(1, 10_001):
        k_crit = binom.ppf(1 - alpha, n, 0.5)
        pw     = 1 - binom.cdf(k_crit - 1, n, p_ij)
        if pw >= power:
            return n, int(k_crit), p_ij, pw

    return None, None, p_ij, None


n_min, k_crit, p_ij, achieved_power = min_comparisons_power(beta3, beta5)
print(f"\nn_min = {n_min}")
print(f"Seuil rejet    = {k_crit} victoires")
print(f"Puissance réelle = {achieved_power:.3f}")


mask = (
    ((df[COL_A] == model3) & (df[COL_B] == model5)) |
    ((df[COL_A] == model5) & (df[COL_B] == model3))
)
df_pair = df[mask].copy()
n_obs   = len(df_pair)


print(f"Paire : {model3}  vs  {model5}")
print(f"Comparaisons directes observées : {n_obs}")
print(f"Minimum requis (puissance 80%)  : {n_min}")

if n_obs == 0:
    print("\n Ces deux modèles ne se sont jamais affrontés directement dans les données.")
    print(f"  Selon le modèle : P({model3} bat {model5}) = {p_ij:.3f}")
elif n_obs >= n_min:
    wins_3 = (df_pair[COL_WINNER] == model3).sum()
    wins_5 = (df_pair[COL_WINNER] == model5).sum()
    ties   = df_pair["both_equal"].fillna(False).astype(bool).sum()
    print(f"\nCondition satisfaite ({n_obs} ≥ {n_min})")
    print(f"  Victoires {model3:<30} : {wins_3}")
    print(f"  Victoires {model5:<30} : {wins_5}")
    print(f"  Ex-æquo                              : {ties}")
else:
    print(f"\nDonnées insuffisantes ({n_obs} < {n_min} comparaisons requises)")


Nombre de votes dans le top20 : 30119

Classement global (top 20) :
 rank                      model      beta
    1        mistral-medium-2508  0.547605
    2           gemini-2.0-flash  0.303831
    3                gemma-3-27b  0.257659
    4                gemma-3-12b  0.012089
    5          claude-4-5-sonnet  0.000000
    6                  command-a -0.095134
    7                 gemma-3-4b -0.144055
    8             gemini-1.5-pro -0.321060
    9         mistral-large-2411 -0.340401
   10               gpt-4.1-mini -0.344739
   11              llama-4-scout -0.377595
   12              llama-3.3-70b -0.589571
   13     gpt-4o-mini-2024-07-18 -0.665974
   14                      phi-4 -0.747482
   15 ministral-8b-instruct-2410 -0.892674
   16    hermes-3-llama-3.1-405b -0.925049
   17             llama-3.1-405b -0.978472
   18               llama-3.1-8b -1.109712
   19 qwen2.5-coder-32b-instruct -1.172816
   20          mistral-nemo-2407 -1.714266

Rang 3 : gemma-3-27b  β = 0


### Problème posé
Combien de comparaisons directes entre gemma-3-27b et claude-4-5-sonnet
faudrait-il observer pour conclure statistiquement que l'un est meilleur ?
 
### Structure du test
Chaque comparaison directe est modélisée comme un tirage de Bernoulli
avec p = 0.564 (probabilité que gemma batte claude, estimée par BT).
 
* H0 : p = 0.5  (les deux sont équivalents)
* H1 : p = 0.564 (gemma est meilleur, selon le modèle)
 
alpha = 0.05  : 5% de risque de rejeter H0 à tort

Puissance = 80% : 80% de chance de détecter la vraie différence si elle existe

p = 0.564 : effet à détecter, issu des scores Bradley-Terry

### Résultat
* n_min          = 348 comparaisons
* Seuil rejet    = 189 victoires
* Puissance réelle = 0.800
 
Comparaisons directes observées : 118

Minimum requis                  : 348

=> Données insuffisantes (118 < 348)
 
Ces deux modèles se sont affrontés directement 118 fois dans les données
mais c'est insuffisant. Il en faudrait 348, soit 3 fois plus que ce qui a
été observé.
 
C'est le point clé : BT permet d'inférer P(gemma bat claude) = 0.564 par
transitivité (via les matchs communs avec d'autres modèles), mais il faudrait
348 matchs directs pour le confirmer statistiquement. C'est exactement là
qu'intervient la transitivité stochastique : elle permet de faire ce pont
entre "118 matchs observés ne suffisent pas" et "on peut quand même estimer
qui est meilleur".
 
### Est-ce que gemma bat claude ?

Nous ne savons pas avec certitude, mais le modèle BT dit que probablement oui.

P = 0.564 => gemma est probablement meilleur.

Mais cette estimation vient d'inférences indirectes et de seulement 118
confrontations directes, loin des 348 requises. Sans ces 348 matchs, il est
impossible de rejeter statistiquement l'hypothèse qu'ils sont équivalents.
 
Le classement BT est cohérent et transitif, mais sa valeur prédictive reste
une extrapolation. L'analyse de puissance quantifie précisément combien de
données il faudrait pour transformer cette extrapolation en certitude statistique.
 
### Lien entre la transitivité stochastique et BT
 
  * gemma    bat  mistral-medium   avec P = 0.68   (matchs directs observés)
  * mistral  bat  claude           avec P = 0.61   (matchs directs observés)
 
  Transitivité stochastique dit :

  P(gemma bat claude) >= max(0.68, 0.61) = 0.68   (borne garantie)

  Bradley-Terry calcule : P = 0.564               (estimation précise)
 
Note : l'estimation BT (0.564) est inférieure à la borne garantie par la
transitivité (0.68) car BT intègre l'ensemble des matchs observés, pas
seulement ce chemin indirect. Les 118 confrontations directes entre gemma
et claude tempèrent l'estimation globale.
 
 

## Davidson
Bradley-Terry classique suppose des résultats binaires : victoire ou défaite. Mais Compar:IA produit aussi des ex-æquo (l'utilisateur ne préfère aucun modèle). Les ignorer ou les compter comme demi-victoires biaise les estimations.

Modèle Davidson (1970) — étend Bradley-Terry aux ex-æquo.
    P(i bat j)  = π_i  / (π_i + π_j + ν√(π_i·π_j))
    P(ex-æquo)  = ν√(π_i·π_j) / (π_i + π_j + ν√(π_i·π_j))
    Paramètre supplémentaire ν ≥ 0 : propension à l'ex-æquo.

In [3]:
def detect_ties(df_pairs):
    def _is_tie(row):
        w = str(row[COL_WINNER]).strip().lower()
        if pd.isna(row[COL_WINNER]) or w in ("", "nan", "tie", "both", "equal"):
            return True
        return w not in (str(row[COL_A]).strip().lower(),
                         str(row[COL_B]).strip().lower())
    return df_pairs.apply(_is_tie, axis=1)
 
def fit_davidson(df_pairs):
    is_tie = detect_ties(df_pairs)
    models = sorted(set(df_pairs[COL_A]) | set(df_pairs[COL_B]))
    idx    = {m: i for i, m in enumerate(models)}
    n      = len(models)
 
    wins = np.zeros((n, n))   # wins[i,j] = nb fois i bat j
    ties = np.zeros((n, n))   # ties[i,j] = nb ex-æquo entre i et j
 
    for (_, row), tie in zip(df_pairs.iterrows(), is_tie):
        ia, ib = idx[row[COL_A]], idx[row[COL_B]]
        if tie:
            ties[ia, ib] += 1
            ties[ib, ia] += 1
        elif row[COL_WINNER] == row[COL_A]:
            wins[ia, ib] += 1
        elif row[COL_WINNER] == row[COL_B]:
            wins[ib, ia] += 1
 
    def neg_log_lik(params):
        beta, theta = params[:n], params[n]   # theta = log(ν)
        nll = 0.0
        for i in range(n):
            for j in range(i + 1, n):
                denom = (np.exp(beta[i]) + np.exp(beta[j])
                         + np.exp(theta) * np.exp((beta[i] + beta[j]) / 2))
                denom = max(denom, 1e-300)
                if wins[i, j] > 0:
                    nll -= wins[i, j] * (beta[i] - np.log(denom))
                if wins[j, i] > 0:
                    nll -= wins[j, i] * (beta[j] - np.log(denom))
                if ties[i, j] > 0:
                    nll -= ties[i, j] * (theta + (beta[i]+beta[j])/2 - np.log(denom))
        return nll
 
    p0     = np.zeros(n + 1)                         # init β=0, θ=0 (ν=1)
    bounds = [(0, 0)] + [(None, None)] * (n - 1) + [(None, None)]
    res    = minimize(neg_log_lik, p0, method="L-BFGS-B",
                      bounds=bounds, options={"maxiter": 5000})
 
    nu     = np.exp(res.x[n])
    result = (
        pd.DataFrame({"model": models, "beta_davidson": res.x[:n]})
        .sort_values("beta_davidson", ascending=False)
        .reset_index(drop=True)
    )
    result["rank_davidson"] = result.index + 1
    return result, nu
 

 
ranking_dav, nu = fit_davidson(df_top20)
print(f"\nClassement Davidson (avec ex-æquo, ν = {nu:.4f}) :")
print(ranking_dav[["rank_davidson", "model", "beta_davidson"]].to_string(index=False))
 
 
 
for label, rk, beta_col in [("Davidson", ranking_dav, "beta_davidson")]:
    r3 = rk[rk[rk.columns[rk.columns.str.startswith("rank")][0]] == 3].iloc[0]
    r5 = rk[rk[rk.columns[rk.columns.str.startswith("rank")][0]] == 5].iloc[0]
    print(f"Puissance — modèle {label}")
    print(f"  Rang 3 : {r3['model']}  β = {r3[beta_col]:.4f}")
    print(f"  Rang 5 : {r5['model']}  β = {r5[beta_col]:.4f}")
    n_min, k_crit, p_ij, pw = min_comparisons_power(r3[beta_col], r5[beta_col])
    print(f"  n_min = {n_min}  |  seuil = {k_crit}  |  puissance = {pw:.3f}")
 


Classement Davidson (avec ex-æquo, ν = 1.0361) :
 rank_davidson                      model  beta_davidson
             1        mistral-medium-2508       0.529742
             2           gemini-2.0-flash       0.360006
             3                gemma-3-27b       0.271453
             4                gemma-3-12b       0.036292
             5          claude-4-5-sonnet       0.000000
             6                  command-a      -0.079678
             7                 gemma-3-4b      -0.135696
             8             gemini-1.5-pro      -0.300763
             9               gpt-4.1-mini      -0.316604
            10         mistral-large-2411      -0.334526
            11              llama-4-scout      -0.372751
            12              llama-3.3-70b      -0.580823
            13     gpt-4o-mini-2024-07-18      -0.666986
            14                      phi-4      -0.743335
            15 ministral-8b-instruct-2410      -0.891209
            16    hermes-3-llama-3.1-4

L'écart β = 0.2715 − 0.0000 = 0.2715, c'est un écart faible. Comparé aux résultats BT où l'écart était 0.4703 − 0.0044 = 0.4659, presque le double. Davidson a donc rapproché ces deux modèles en rang réel.

Davidson a redistribué le "crédit" des ex-æquo. Claude-4-5-sonnet tire probablement plus souvent des matchs nuls que gemma-3-27b, ces nuls comptaient pour rien dans BT, ils lui donnent maintenant du crédit dans Davidson.

Il faut 310 matchs directs pour distinguer statistiquement ces deux modèles, contre 102 dans BT.

p = 0.5674 est beaucoup plus proche de 0.5 que p = 0.614. Plus deux modèles sont proches, plus il faut de données pour prouver qu'ils sont différents, c'est la logique de toute analyse de puissance.

Le classement Davidson révèle une structure asymétrique : une tête concentré (top 3 sur 0.26 points de β) difficile à départager statistiquement, et une suite de classement très détachée (plus de 2 points d'écart total), confirmant que la mesure est discriminante pour les modèles extrêmes mais statistiquement fragile pour les modèles proches du centre.

La différence entre les deux modèles est faible (348 vs 310 matchs requis),
ce qui confirme que les ex-æquo n'ont pas un impact massif sur cette paire.
Les deux modèles se départagent de façon similaire avec ou sans prise en
compte des matchs nuls.

## Conclusion partie classement BT

Les deux modèles sont proches, et Davidson le confirme en redistribuant le crédit des ex-æquo. Contrairement au cas précédent, Davidson augmente légèrement l'écart β ici (0.2715 vs 0.2577 pour BT) ce qui suggère que gemma bénéficie des matchs nuls plus que claude. Il faudrait 310 confrontations directes pour le prouver statistiquement, alors que seulement 118 ont été observées. Les ex-æquo ne sont pas du bruit à ignorer, ils portent une information sur la proximité réelle des modèles et sur lequel des deux tire davantage profit des matchs nuls.